[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Реализуйте **2D convolution** с базовыми тензорными операциями. Фильтр скользит по высоте и ширине изображения; в каждой позиции берётся локальный patch, умножается поэлементно на kernel и суммируется по входным каналам и spatial-размерностям.

### Формы тензоров
- input: `(B, C_in, H, W)`;
- weight: `(C_out, C_in, kH, kW)` — отдельный kernel для каждого выходного канала;
- bias: `(C_out,)`, broadcast на batch и spatial positions;
- output: `(B, C_out, H_out, W_out)`.

Для dilation=1 и симметричного padding:

$$H_{out}=\left\lfloor\frac{H+2p-kH}{s}\right\rfloor+1, \qquad W_{out}=\left\lfloor\frac{W+2p-kW}{s}\right\rfloor+1$$

### Что вычисляется в одной позиции
Patch формы `(C_in, kH, kW)` умножается на kernel выбранного `C_out`. Сумма всех произведений даёт один output scalar, затем добавляется bias. PyTorch фактически выполняет cross-correlation: kernel не переворачивается по spatial axes.

### Stride и padding
`padding` добавляет нули вокруг изображения до извлечения patches. `stride` задаёт сдвиг окна; stride больше единицы уменьшает output resolution и пропускает промежуточные позиции.

### Контракт
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### План реализации
1. При необходимости добавьте zero-padding.
2. Вычислите `H_out` и `W_out`.
3. Извлекайте patches вложенными индексами или через unfold.
4. Одновременно учитывайте все `C_in`, затем формируйте все `C_out`.
5. Добавьте bias с правильным broadcasting.

### Ограничения
- не используйте `F.conv2d` или `nn.Conv2d`;
- поддержите `stride` и `padding`;
- `F.pad` для zero-padding разрешён;
- сохраняйте autograd для input и weight.

### Быстрые самопроверки
- kernel `3×3`, padding `0`, stride `1` уменьшает `8×8` до `6×6`;
- padding `1` сохраняет spatial size для нечётного kernel `3×3`;
- stride `2` уменьшает число позиций;
- результат совпадает с `F.conv2d` после самостоятельного решения;
- backward создаёт gradients для input и weight.

### Типичные ошибки
- потерянная сумма по `C_in`;
- перепутанные `C_in` и `C_out`;
- неверная формула output shape;
- сдвиг окна на `1` вместо `stride`;
- kernel переворачивается как в математической convolution.

### Официальные материалы и примеры
- [`torch.nn.functional.conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.conv2d.html) — контракт, формы и эталонный пример;
- [`torch.nn.functional.unfold`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.unfold.html) — извлечение sliding blocks;
- [`Tensor.unfold`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.unfold.html) — простой пример окон по одной оси.

### Вопросы для самопроверки
1. Почему output channel суммирует вклад всех input channels?
2. Как padding и stride влияют на output shape?
3. Почему weight имеет форму `(C_out, C_in, kH, kW)`?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    pass  # extract patches, apply kernel, handle stride/padding

In [ ]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')